In [50]:
# Importations
import os
import psycopg
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.functions import round,col, dayofmonth,month,year,quarter,when 
from pyspark.sql import functions as F

In [51]:
# Chargement des variables d'environnement (.env)
load_dotenv()


True

In [52]:
# Paramètres de connexion et chemins
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

JDBC_URL = f"jdbc:postgresql://{DB_HOST}:{DB_PORT}/{DB_NAME}"
JDBC_PROPS = {
    "user": DB_USER,
    "password": DB_PASSWORD,
    "driver": "org.postgresql.Driver"
}

In [53]:
# Initialisation de la session Spark
spark = SparkSession.builder \
    .appName("WindPowerSilverTransformation") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

In [54]:
# Récupérer les tables windturbinepower_cleaned et weatherforecastapi_clean du schéma silver dans des DataFrames Spark

#  Récupération des données de la table silver.windpowerturbine_cleaned
windturbinepower_cleaned_df = spark.read.jdbc(
    url=JDBC_URL,
    table="silver.windpowerturbine_cleaned",
    properties=JDBC_PROPS
)

# Récupération des données de la table silver.weatherforecastapi_cleaned
weatherforecastapi_cleaned_df = spark.read.jdbc(
    url=JDBC_URL,
    table="silver.weatherforecastapi_cleaned",
    properties=JDBC_PROPS
)

In [55]:
# Préparation des colonnes et jointure pour construire la table de synthèse
wind_selected_df = windturbinepower_cleaned_df.select(
    "production_id",
    "date",
    F.date_format(col("time"), "HH:mm:ss").alias("time"),
    "latitude",
    "longitude",
    "region",
    "turbine_name",
    "capacity",
    "status",
    "responsible_department",
    "energy_produced",
    "wind_direction",
    "day",
    "month",
    "quarter",
    "year",
    "hour_of_day",
    "minute_of_hour",
    "second_of_minute",
    "time_period"
)

weather_selected_df = weatherforecastapi_cleaned_df.select(
    "weather_id",
    "date",
    F.date_format(col("time"), "HH:mm:ss").alias("time"),
    "region",
    "region_name",
    "wind_speed_100m",
    "wind_gusts_10m",
    "temperature_2m",
    "pressure_msl",
    "precipitation"
)

enriched_df = (
    wind_selected_df.alias("w")
    .join(
        weather_selected_df.alias("m"),
        on=["date", "time", "region"],
        how="left"
    )
    .select(
        col("w.production_id").alias("production_id"),
        col("m.weather_id").alias("weather_id"),
        col("w.date").alias("date"),
        col("w.time").alias("time"),
        col("w.latitude").alias("latitude"),
        col("w.longitude").alias("longitude"),
        col("w.region").alias("region"),
        col("m.region_name").alias("region_name"),
        col("w.turbine_name").alias("turbine_name"),
        col("w.capacity").alias("capacity"),
        col("w.status").alias("status"),
        col("w.responsible_department").alias("responsible_department"),
        col("w.energy_produced").alias("energy_produced"),
        col("m.wind_speed_100m").alias("wind_speed_100m"),
        col("m.wind_gusts_10m").alias("wind_gust_10m"),
        col("w.wind_direction").alias("wind_direction"),
        col("m.temperature_2m").alias("temperature_2m"),
        col("m.pressure_msl").alias("pressure_msl"),
        col("m.precipitation").alias("precipitation"),
        col("w.day").alias("day"),
        col("w.month").alias("month"),
        col("w.quarter").alias("quarter"),
        col("w.year").alias("year"),
        col("w.hour_of_day").alias("hour_of_day"),
        col("w.minute_of_hour").alias("minute_of_hour"),
        col("w.second_of_minute").alias("second_of_minute"),
        col("w.time_period").alias("time_period")
    )
)

enriched_df.show(5)

+-------------+----------+----------+--------+--------+---------+--------+--------------------+------------+--------+------+----------------------+---------------+---------------+-------------+--------------+--------------+------------+-------------+---+-----+-------+----+-----------+--------------+----------------+-----------+
|production_id|weather_id|      date|    time|latitude|longitude|  region|         region_name|turbine_name|capacity|status|responsible_department|energy_produced|wind_speed_100m|wind_gust_10m|wind_direction|temperature_2m|pressure_msl|precipitation|day|month|quarter|year|hour_of_day|minute_of_hour|second_of_minute|time_period|
+-------------+----------+----------+--------+--------+---------+--------+--------------------+------------+--------+------+----------------------+---------------+---------------+-------------+--------------+--------------+------------+-------------+---+-----+-------+----+-----------+--------------+----------------+-----------+
|         

In [ ]:
# Création de la table de synthèse silver.windturbinepower_enriched
create_table_sql = """
CREATE TABLE IF NOT EXISTS silver.windturbinepower_enriched (
    prod_enriched_id BIGSERIAL PRIMARY KEY,
    production_id INTEGER,
    weather_id BIGINT,
    date DATE,
    time TIME,
    latitude NUMERIC(9,6),
    longitude NUMERIC(9,6),
    region VARCHAR(100),
    region_name VARCHAR(255),
    turbine_name VARCHAR(100),
    capacity INTEGER,
    status VARCHAR(50),
    responsible_department VARCHAR(100),
    energy_produced NUMERIC(12,2),
    wind_speed_100m NUMERIC(6,2),
    wind_gust_10m NUMERIC(6,2),
    wind_direction VARCHAR(50),
    temperature_2m NUMERIC(5,2),
    pressure_msl NUMERIC(7,2),
    precipitation NUMERIC(8,2),
    day INTEGER,
    month INTEGER,
    quarter INTEGER,
    year INTEGER,
    hour_of_day INTEGER,
    minute_of_hour INTEGER,
    second_of_minute INTEGER,
    time_period VARCHAR(20)
);
"""

with psycopg.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    autocommit=True
) as conn:
    with conn.cursor() as cur:
        cur.execute("CREATE SCHEMA IF NOT EXISTS silver;")
        cur.execute(create_table_sql)
        print("Table silver.windturbinepower_enriched créée")

Table silver.windturbinepower_enriched créée


In [57]:
# Insertion anti-doublon simple puis vérification
total_count = enriched_df.count()
print(f"Lignes enrichies à traiter: {total_count}")

try:
    existing_df = spark.read.jdbc(
        url=JDBC_URL,
        table="silver.windturbinepower_enriched",
        properties=JDBC_PROPS
    )
    existing_keys = existing_df.select(
        "production_id",
        "date",
        F.date_format(col("time"), "HH:mm:ss").alias("time")
    )
    df_to_insert = enriched_df.join(
        existing_keys,
        on=["production_id", "date", "time"],
        how="left_anti"
    )
except Exception:
    df_to_insert = enriched_df

new_count = df_to_insert.count()
print(f"Nouvelles lignes à insérer: {new_count}")

if new_count > 0:
    df_final = df_to_insert \
        .withColumn("time", F.to_timestamp(col("time"), "HH:mm:ss")) \
        .orderBy("date", "time", "region", "turbine_name")

    df_final.write.jdbc(
        url=JDBC_URL,
        table="silver.windturbinepower_enriched",
        mode="append",
        properties=JDBC_PROPS
    )
    print(f"{new_count} lignes insérées avec succès")
else:
    print("Aucune donnée à insérer")

final_count = spark.read.jdbc(
    url=JDBC_URL,
    table="silver.windturbinepower_enriched",
    properties=JDBC_PROPS
).count()
print(f"Total en base: {final_count} lignes")

Lignes enrichies à traiter: 1296
Nouvelles lignes à insérer: 0
Aucune donnée à insérer
Total en base: 1296 lignes
